# DeepRL Monopoly — PPO hybrid, Builder+DealMaker+Hoarder self-play

Separate from the DDQN notebooks (`train_colab.ipynb`, `train_colab_pro.ipynb`) —
own checkpoint path/seed. PPO trains far faster than DDQN in local testing
(~2.6s/game vs ~15-25s/game) and produced our first observed win in a 4-game
smoke test, so this is the priority run.

ASU is not used anywhere in this notebook — no teacher, no labels, no distillation.

## 1. Mount Drive (own checkpoint folder)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt_ppo'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [ ]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

## 3. Check GPU + torch

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime > Change runtime type > GPU, then re-run this cell.')

## 4. Train (PPO, hybrid buy+build+trade+accept heuristics)

Opponent table: `TheBuilder + TheDealMaker + TheHoarder`. `--checkpoint-every 100`
writes a resumable checkpoint so a disconnect just needs `--resume` (cell 6) to
continue from the same point — nothing is lost.

In [ ]:
import os
OUT = f"{CHECKPOINT_DIR}/ppo_hybrid_mix_v1.pt"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ppo --games 5000 --device auto --seed 99 \
  --checkpoint-every 100 \
  --out "{OUT}"

## 5. Analyze the per-game log

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/analyze_run.py "{OUT.rsplit('.', 1)[0]}_games.csv" --window 100

## 6. Resume after a disconnect (same --out path, --resume flag)

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ppo --games 20000 --device auto --seed 99 --resume \
  --checkpoint-every 100 \
  --out "{OUT}"

## 7. Quick eval against Builder + DealMaker + Hoarder after training

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/play_game.py \
  --algo ppo --players 4 --model "{OUT}"